# 🎬 Test đếm trên VIDEO THẬT — nhiều cảnh + query khó

Video công khai, tải trực tiếp (Roboflow supervision + Pexels). Đã **sửa nhãn sai**
(bỏ video nước chảy, chuyển video giao thông về đúng bài xe). Nhấn mạnh **đếm sản
phẩm dây chuyền** với **nhiều query khó** để test khả năng mô tả ngôn ngữ tự nhiên.

| Bài toán | Video | Query test |
|----------|-------|-----------|
| 🚗 Phương tiện | 4 (cao tốc, giao lộ, phố) | car/truck/bus… |
| 📦 Dây chuyền | 6 (chiết chai, kiện hàng, đóng gói) | chai/thùng/sản phẩm lỗi… |
| 🚶 Người | 4 (lối đi, ga tàu, siêu thị, quảng trường) | người đeo balo, đẩy xe… |


## 1) Tải code + cài thư viện


In [ ]:
%cd /kaggle/working
!rm -rf VisionOS
!git clone -q https://github.com/nguyendinhhuyht20032004-ai/VisionOS.git
%cd VisionOS/VisionOS
!git checkout -q claude/rebuild-visionos-codebase-tgg0mf
!git pull -q origin claude/rebuild-visionos-codebase-tgg0mf
!pip install -q ultralytics 'supervision>=0.21' opencv-python-headless


## 2) Xem catalog (video + query khó gợi ý cho mỗi video)


In [ ]:
!python run_scenarios.py --list


## 3) 🚗 Đếm PHƯƠNG TIỆN (YOLO, nhanh) — 4 góc


In [ ]:
!python run_scenarios.py --task vehicles --max-frames 300


## 4) 🚶 Đếm NGƯỜI (YOLO, nhanh) — 4 cảnh


In [ ]:
!python run_scenarios.py --task people --max-frames 300


## 5) 📦 Đếm SẢN PHẨM — chai trên chuyền (YOLO, nhanh)
`bottle` là lớp COCO → YOLO đếm được, không cần LocateAnything.


In [ ]:
!python run_scenarios.py --task conveyor --only milk --max-frames 300


## 6) 📦🧠 Đếm SẢN PHẨM với QUERY KHÓ (open-vocab LocateAnything)
Test **nhiều query khó** trên các video dây chuyền — kiểm tra model hiểu mô tả đến đâu.
Tự cài transformers==4.57.1 (auto-pin) + attn=sdpa an toàn cho T4. **Chạy chậm.**

- `--all-queries`: chạy tất cả query gợi ý sẵn của mỗi video.
- Hoặc tự ra đề: `--queries "chai nước ngọt,thùng carton,sản phẩm bị móp"`.


In [ ]:
# Query khó gợi ý sẵn cho từng video dây chuyền (chai không nắp, thùng móp, sản phẩm lỗi…):
!python run_scenarios.py --task conveyor --all-queries --max-frames 80


In [ ]:
# Hoặc tự ra đề query của bạn (mọi video dây chuyền chạy các query này):
!python run_scenarios.py --task conveyor --queries "cardboard box,plastic bottle,a damaged package,the largest item" --max-frames 80


## 7) 🧠 Query khó cho NGƯỜI / XE (open-vocab)
Ví dụ chỉ đếm 'người đeo ba lô', 'xe tải màu trắng'…


In [ ]:
!python run_scenarios.py --task people --only walk --queries "a person wearing a backpack,a person carrying a bag" --max-frames 120


---
### Đọc scorecard
- Có thêm cột **query** → so kết quả giữa các prompt dễ/khó trên cùng video.
- **det/frame** thấp ở query khó = model chưa hiểu mô tả đó → thử diễn đạt khác.
- Video sai nhãn/đếm lệch: sửa trong `recognition/video_catalog.py` (mỗi video có 'mẹo').
- Thêm video: thêm 1 `VideoScenario(asset|url|pexels_id, queries=(...))` vào catalog.
